In [107]:
from openai import OpenAI
from typing import Type
import os
import re
import base64
import pandas as pd
from datetime import datetime
from pdf2image import convert_from_path
from dotenv import load_dotenv
from io import BytesIO
from pydantic import BaseModel, Field
import json

import Validity_Functions

In [108]:
pdf_path = "/zfs/projects/students/ltdarc-usf-intern-2025/data/Arizona_Republic_Sun__Dec_17__2000_ (15).pdf"

In [109]:
def pdf_to_b64(pdf_path: str) -> str:
    """ Converts pdf image to b64."""
    pages = convert_from_path(pdf_path, first_page=1, last_page=1, dpi=100)
    buffer = BytesIO()
    pages[0].save(buffer, format="PNG")
    img_bytes = buffer.getvalue()
    img_b64 = base64.b64encode(img_bytes).decode()

    return img_b64

In [110]:
load_dotenv("/zfs/projects/students/ltdarc-usf-intern-2025/.env")

True

In [111]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [112]:
model = "gpt-4o-mini"

In [113]:
role_prompt = "You are a metadata extraction assistant that specializes reading the TV guides of newspapers."


In [114]:
content_prompt = """Look at this newspaper page and extract two fields:
1. The name of the newspaper
2. The publication date (do not include day of week)

Return your answer ONLY as a valid JSON object with the following keys:
  - newspaper
  - date

Example:
{
  "newspaper": "The New York Times",
  "date": "January 5, 2023"
}
"""

In [115]:
class MetaData(BaseModel):
    newspaper_name: str
    publication_date: str
    

In [116]:
def call_llm(b64: str, model: str, role_prompt: str, content_prompt: str, structured_output: Type[BaseModel]) -> BaseModel:
    """Feeds B64 image into LLM and returns structured JSON as MetaData."""
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    response = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": [{"type": "input_text", "text": role_prompt}]},
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": content_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{b64}",
                        "detail":"low"
                    },
                ],
            },
        ],
        text_format=structured_output,
    )

    return response.output_parsed
    

In [117]:
b64 = pdf_to_b64(pdf_path)
call_llm(b64, model, role_prompt, content_prompt, MetaData)

MetaData(newspaper_name='Arizona Republic', publication_date='December 17, 2000')

In [118]:
data_path='/zfs/projects/students/ltdarc-usf-intern-2025/data'

index = Validity_Functions.make_index(data_path)

In [126]:
def extract_meta_from_filename(pdf_path: str) -> dict:
    """
    Extracts newspaper name and date from a PDF filename.
    Handles embedded weekday names (Sun, Monday, etc.).
    Example filename:
      Arizona_Republic_Sun__Dec_17__2000_ (15).pdf
    Returns:
      {'true_name': 'Arizona Republic', 'true_date': 'December 17, 2000'}
    """
    filename = os.path.basename(pdf_path)
    filename = filename.replace(".pdf", "").strip()
    filename = re.sub(r" ?\(\d+\)$", "", filename)  # remove '(15)' etc.

    # Normalize underscores
    filename = filename.replace("__", "_").replace("__", "_")  # collapse double underscores

    # Remove weekdays anywhere in the string
    weekdays_pattern = (
        r"(_|-)?(?i:"
        r"Mon(day)?|Tue(s|sday)?|Wed(nesday)?|Thu(r|rs|rsday)?|Fri(day)?|Sat(urday)?|Sun(day)?"
        r")(_|-)?"
    )
    filename = re.sub(weekdays_pattern, "_", filename)
    filename = re.sub(r"_+", "_", filename)  # collapse multiple underscores again

    # Split parts
    parts = [p for p in filename.split("_") if p.strip()]

    # Find month, day, year tokens
    month_map = {m[:3]: m for m in [
        "January","February","March","April","May","June",
        "July","August","September","October","November","December"
    ]}

    month, day, year = None, None, None
    name_parts = []

    for token in parts:
        short = token[:3].capitalize()
        if short in month_map:
            month = month_map[short]
        elif token.isdigit():
            if len(token) == 4:
                year = token
            elif len(token) <= 2:
                day = token
        else:
            name_parts.append(token)

    true_name = " ".join(name_parts).strip()
    true_date = None
    if month and day and year:
        try:
            dt = datetime.strptime(f"{month} {int(day)}, {year}", "%B %d, %Y")
            true_date = dt.strftime("%B %d, %Y")
        except Exception:
            true_date = f"{month} {day}, {year}"

    return {"true_name": true_name, "true_date": true_date}

In [127]:
results = []

for idx, row in index.iterrows():
    pdf_path = row["pdf_files"]
    b64 = pdf_to_b64(pdf_path)

    llm_meta = call_llm(b64, model, role_prompt, content_prompt, MetaData)
    meta_truth = extract_meta_from_filename(pdf_path)

    if llm_meta.newspaper_name == meta_truth["true_name"]:
        name_match = True
    else:
        name_match = False

    if llm_meta.publication_date == meta_truth["true_date"]:
        date_match = True
    else:
        date_match = False

    results.append({
        "Index": idx,
        "LLM_Newspaper_Name": llm_meta.newspaper_name,
        "LLM_Newspaper_Date": llm_meta.publication_date,
        "Actual_Name": meta_truth["true_name"],
        "Actual_Date": meta_truth["true_date"],
        "Name_Match": name_match,
        "Date_Match": date_match,
    })

comparison_df = pd.DataFrame(results)

In [128]:
print(comparison_df)

    Index         LLM_Newspaper_Name  LLM_Newspaper_Date  \
0       0           Arizona Republic   December 17, 2000   
1       1  Austin American-Statesman      August 3, 2014   
2       2  Austin American-Statesman   February 22, 2021   
3       3  Austin American-Statesman       June 12, 2014   
4       4  Austin American-Statesman   November 29, 2014   
5       5            Chicago Tribune        May 28, 1995   
6       6            Chicago Tribune    October 30, 1984   
7       7                 Daily News      April 13, 2008   
8       8      Daily News (New York)      April 25, 2010   
9       9      Daily News (New York)       June 17, 2012   
10     10      Daily News (New York)        June 9, 2013   
11     11      Daily News (New York)      March 11, 2012   
12     12      Daily News (New York)      March 30, 2014   
13     13      Daily News (New York)        May 15, 2016   
14     14                 Daily News    October 14, 2007   
15     15      Daily News (New York)    

In [125]:
comparison_df['Actual_Name']

0              Arizona Republic Sun
1     Austin American Statesman Sun
2     Austin American Statesman Sun
3     Austin American Statesman Sun
4     Austin American Statesman Sun
5               Chicago Tribune Sun
6               Chicago Tribune Sun
7                    Daily News Sun
8                    Daily News Sun
9                    Daily News Sun
10                   Daily News Sun
11                   Daily News Sun
12                   Daily News Sun
13                   Daily News Sun
14                   Daily News Sun
15                   Daily News Sun
16     Honolulu Star Advertiser Sun
17     Honolulu Star Advertiser Sun
18           Messenger Inquirer Sun
19            Statesman Journal Sun
20            Statesman Journal Sun
21            Statesman Journal Sun
22         The Arizona Republic Sun
23              The Californian Sun
24      The Honolulu Advertiser Sun
25      The Honolulu Advertiser Sun
26      The Honolulu Advertiser Sun
27      The Honolulu Adverti